In [29]:
!pip install torch pillow transformers datasets

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached triton-3.6.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 58.3 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 88.9 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 107.7 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 107.7 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 104.3 MB/s  0:00:00m0:00:0100:01
Using cached triton-3.6.0-cp313-cp

In [ ]:
!pip uninstall transformers -y
!pip install "transformers==4.50" accelerate

In [ ]:
%pip uninstall transformers -y
%pip install transformers==4.50

In [ ]:
%pip install -U "accelerate>=0.26.0"

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

image = Image.open("plot1.png").convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What is the median of the 2003 section? If exact value is not visible, estimate and say how you estimated."}
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True)

inputs = processor(
    text=text,
    images=image,
    return_tensors="pt"
).to(model.device)

out = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(out[0], skip_special_tokens=True))


In [ ]:
# wrong value and no explanation

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

image = Image.open("plot1.png").convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What is the maximum value? If exact value is not visible, estimate and say how you estimated."}
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True)

inputs = processor(
    text=text,
    images=image,
    return_tensors="pt"
).to(model.device)

out = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(out[0], skip_special_tokens=True))


In [ ]:
# everything correct except year which is 2004 instead of 2005

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

image = Image.open("plot1.png").convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Is the data increasing or decreasing? Give answers with respect to each country. If correlation is not found, describe the changes."}
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True)

inputs = processor(
    text=text,
    images=image,
    return_tensors="pt"
).to(model.device)

out = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(out[0], skip_special_tokens=True))


In [ ]:
# struggles to describe accurately in specifics when both there is and isn't any clear correlation
# (ex: for Italy, value doesn't remain stable in 2005 - it shoots up)
# (ex: for USA, is also inaccurate because it overall increases, not decreases)

In [ ]:
%pip install pillow

In [ ]:
from huggingface_hub import snapshot_download
repo_id = "ReadingTimeMachine/visual_qa_histograms"
local_dir = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=[
        "example_hists_larger/**",
        "README.md",
    ],
)

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import re

# load csvs
df1 = pd.read_csv("hist_complex_parsed.csv")
df2 = pd.read_csv("hist_larger_parsed.csv")
df = pd.concat([df1, df2], ignore_index=True)

# actual downloaded image folder you found
img_dir = os.path.join(local_dir, "example_hists_larger", "imgs")

# for the larger csv, the image filename should match the basename
df["image_file"] = df["image_path"].apply(lambda p: os.path.basename(p))
df["full_image_path"] = df["image_file"].apply(lambda f: os.path.join(img_dir, f))

print(df[["image_path", "image_file", "full_image_path"]].head())

In [ ]:
idx = 0
test_path = df.iloc[idx]["full_image_path"]
print(test_path)

image = Image.open(test_path).convert("RGB")
display(image)